## Train

Goal: Train MDNRNN on the Car Racing Task

In [2]:
import hashlib
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import torch

from world_models.models.vae import VAE
from world_models.models.mdn_rnn import MDNRNN
from world_models.training.mdn_rnn import mdn_loss
from world_models.data.latents import encode_episode

cwd = Path.cwd().resolve()
repo_root = next(
    p for p in (cwd, *cwd.parents)
    if (p / "src" / "world_models").is_dir()
    and (p / "pyproject.toml").is_file()
)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

# List available checkpoints so you can select the completed run.
for path in sorted((repo_root / "runs").glob("vae_pilot_*/best.pt")):
    print(path.relative_to(repo_root))

runs/vae_pilot_20260910_190359_481752/best.pt


In [4]:
checkpoint_path = (
    repo_root / "runs" / "vae_pilot_20260910_190359_481752" / "best.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

vae = VAE(latent_dim=checkpoint["latent_dim"])
vae.load_state_dict(checkpoint["model_state_dict"])
vae = vae.to(device).eval()
vae.requires_grad_(False)

dataset_dir = Path(checkpoint["dataset_dir"])
manifest = checkpoint["dataset_manifest"]

print("Loaded VAE epoch:", checkpoint["epoch"])
print("Dataset:", dataset_dir)
print("Device:", device)

Loaded VAE epoch: 5
Dataset: /Users/cedric/Repos/world-models-reproduction/data/carracing_pilot_legacy_v1_20260910_185047_641885
Device: mps


In [5]:
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
latent_dir = repo_root / "data" / f"carracing_latents_{run_id}"
latent_dir.mkdir(parents=True, exist_ok=False)

latent_manifest = {
    "status": "encoding",
    "source_dataset": str(dataset_dir),
    "vae_checkpoint": str(checkpoint_path),
    "vae_checkpoint_sha256": hashlib.sha256(
        checkpoint_path.read_bytes()
    ).hexdigest(),
    "latent_dim": checkpoint["latent_dim"],
    "storage_dtype": "float32",
    "episodes": [],
}

manifest_path = latent_dir / "manifest.json"


def save_latent_manifest():
    manifest_path.write_text(
        json.dumps(latent_manifest, indent=2),
        encoding="utf-8",
    )


save_latent_manifest()

for record in manifest["episodes"]:
    with np.load(dataset_dir / record["file"]) as episode:
        mu, logvar = encode_episode(
            vae, episode["observations"]
        )
        actions = episode["actions"].copy()

    assert len(mu) == len(actions) + 1
    assert mu.shape == logvar.shape

    np.savez_compressed(
        latent_dir / record["file"],
        mu=mu,
        logvar=logvar,
        actions=actions,
    )

    # Preserve the original episode split and seed.
    latent_manifest["episodes"].append(dict(record))
    save_latent_manifest()

    print(
        f"{record['file']} | "
        f"{record['split']} | mu={mu.shape}",
        flush=True,
    )

latent_manifest["status"] = "complete"
save_latent_manifest()

print("Saved latent dataset:", latent_dir)

episode_0000.npz | train | mu=(1001, 32)
episode_0001.npz | train | mu=(1001, 32)
episode_0002.npz | train | mu=(1001, 32)
episode_0003.npz | train | mu=(1001, 32)
episode_0004.npz | train | mu=(1001, 32)
episode_0005.npz | train | mu=(1001, 32)
episode_0006.npz | train | mu=(1001, 32)
episode_0007.npz | train | mu=(1001, 32)
episode_0008.npz | train | mu=(1001, 32)
episode_0009.npz | train | mu=(1001, 32)
episode_0010.npz | train | mu=(1001, 32)
episode_0011.npz | train | mu=(1001, 32)
episode_0012.npz | train | mu=(1001, 32)
episode_0013.npz | train | mu=(1001, 32)
episode_0014.npz | train | mu=(1001, 32)
episode_0015.npz | train | mu=(1001, 32)
episode_0016.npz | validation | mu=(1001, 32)
episode_0017.npz | validation | mu=(1001, 32)
episode_0018.npz | validation | mu=(1001, 32)
episode_0019.npz | validation | mu=(1001, 32)
Saved latent dataset: /Users/cedric/Repos/world-models-reproduction/data/carracing_latents_20260910_191353_099643


In [6]:
first_file = latent_manifest["episodes"][0]["file"]

with np.load(latent_dir / first_file) as episode:
    mu = torch.from_numpy(episode["mu"])
    logvar = torch.from_numpy(episode["logvar"])
    actions = torch.from_numpy(episode["actions"])

torch.manual_seed(0)
z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

current_z = z[:-1]
targets = z[1:]

print("Current latents:", current_z.shape)  # [1000, 32]
print("Actions:", actions.shape)            # [1000, 3]
print("Next latents:", targets.shape)       # [1000, 32]

Current latents: torch.Size([1000, 32])
Actions: torch.Size([1000, 3])
Next latents: torch.Size([1000, 32])


In [7]:
from torch.utils.data import Dataset, DataLoader


class LatentEpisodes(Dataset):
    def __init__(self, directory, manifest, split):
        self.episodes = []

        for record in manifest["episodes"]:
            if record["split"] != split:
                continue

            with np.load(directory / record["file"]) as data:
                self.episodes.append({
                    key: torch.from_numpy(data[key].copy()).float()
                    for key in ("mu", "logvar", "actions")
                })

        if not self.episodes:
            raise ValueError(f"No episodes for split: {split}")

    def __len__(self):
        return len(self.episodes)

    def __getitem__(self, index):
        return self.episodes[index]


train_data = LatentEpisodes(latent_dir, latent_manifest, "train")
val_data = LatentEpisodes(latent_dir, latent_manifest, "validation")

train_loader = DataLoader(
    train_data,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(0),
)
val_loader = DataLoader(
    val_data,
    batch_size=2,
    shuffle=False,
    num_workers=0,
)

print("Training episodes:", len(train_data))
print("Validation episodes:", len(val_data))

Training episodes: 16
Validation episodes: 4


In [8]:
torch.manual_seed(0)

memory = MDNRNN(
    latent_dim=32,
    action_dim=3,
    hidden_dim=256,
    num_components=5,
).to(device)

optimizer = torch.optim.Adam(memory.parameters(), lr=1e-3)

memory_history = []
best_val_nll = float("inf")

run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
memory_run_dir = repo_root / "runs" / f"memory_pilot_{run_id}"
memory_run_dir.mkdir(parents=True, exist_ok=False)

In [9]:
def run_memory_epoch(loader, optimizer=None):
    training = optimizer is not None
    memory.train(training)

    # Local CPU generator: reproducible validation samples.
    validation_rng = torch.Generator().manual_seed(12345)

    total_nll = 0.0
    total_transitions = 0

    with torch.set_grad_enabled(training):
        for batch in loader:
            mu = batch["mu"]
            logvar = batch["logvar"]

            if training:
                epsilon = torch.randn_like(mu)
            else:
                epsilon = torch.randn(
                    mu.shape,
                    generator=validation_rng,
                    dtype=mu.dtype,
                )

            # Sample once before slicing to preserve overlap.
            z = (mu + torch.exp(0.5 * logvar) * epsilon).to(device)
            actions = batch["actions"].to(device)

            # Each batch contains unrelated complete episodes.
            logits, means, log_std, _ = memory(
                z[:, :-1],
                actions,
                state=None,
            )

            loss = mdn_loss(
                logits, means, log_std, z[:, 1:]
            )

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite memory loss.")

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    memory.parameters(), max_norm=1.0
                )
                optimizer.step()

            transitions = actions.shape[0] * actions.shape[1]
            total_nll += loss.item() * transitions
            total_transitions += transitions

    return total_nll / total_transitions

In [10]:
for _ in range(10):
    epoch = len(memory_history) + 1

    train_nll = run_memory_epoch(train_loader, optimizer)
    val_nll = run_memory_epoch(val_loader)

    memory_history.append({
        "epoch": epoch,
        "train_nll": train_nll,
        "validation_nll": val_nll,
    })

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": memory.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "model_config": {
            "latent_dim": 32,
            "action_dim": 3,
            "hidden_dim": 256,
            "num_components": 5,
        },
        "latent_dir": str(latent_dir),
        "latent_manifest": latent_manifest,
        "history": memory_history,
        "training_config": {
            "learning_rate": 1e-3,
            "batch_size": 2,
            "gradient_clip_norm": 1.0,
            "validation_sampling_seed": 12345,
        },
    }

    torch.save(checkpoint, memory_run_dir / "last.pt")

    if val_nll < best_val_nll:
        best_val_nll = val_nll
        torch.save(checkpoint, memory_run_dir / "best.pt")

    (memory_run_dir / "history.json").write_text(
        json.dumps(memory_history, indent=2),
        encoding="utf-8",
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train NLL={train_nll:.3f} | "
        f"validation NLL={val_nll:.3f}",
        flush=True,
    )

print("Saved to:", memory_run_dir)

Epoch 01 | train NLL=43.856 | validation NLL=42.118
Epoch 02 | train NLL=39.280 | validation NLL=39.151
Epoch 03 | train NLL=36.402 | validation NLL=35.744
Epoch 04 | train NLL=33.621 | validation NLL=33.537
Epoch 05 | train NLL=31.781 | validation NLL=32.281
Epoch 06 | train NLL=30.911 | validation NLL=31.841
Epoch 07 | train NLL=30.406 | validation NLL=31.820
Epoch 08 | train NLL=30.421 | validation NLL=31.428
Epoch 09 | train NLL=29.849 | validation NLL=31.232
Epoch 10 | train NLL=29.758 | validation NLL=31.146
Saved to: /Users/cedric/Repos/world-models-reproduction/runs/memory_pilot_20260910_191458_564311
